# CMS Medicare Part D â€” Lenalidomide Geographic Map

**Prerequisite:** Run `00_build_database.ipynb` first.

This notebook maps lenalidomide prescribing rates across U.S. states, normalized by Part D enrollment to produce a per-capita metric. Raw claim counts reflect population size; per-capita rates reveal true geographic variation in access and prescribing.

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: Build Per-Capita Table

Join lenalidomide state-level claims to Part D enrollment counts.
Aggregate brand and generic rows to one row per state, then calculate
claims per 100,000 Part D enrollees.

In [2]:
df = pd.read_sql_query("""
    SELECT
        p.Prscrbr_Geo_Desc AS state,
        e.BENE_STATE_ABRVTN AS state_abbr,
        SUM(p.Tot_Clms) AS tot_clms,
        SUM(p.Tot_Benes) AS tot_benes,
        ROUND(SUM(p.Tot_Drug_Cst), 0) AS tot_cost,
        e.PRSCRPTN_DRUG_TOT_BENES AS part_d_enrollees,
        ROUND(SUM(p.Tot_Clms) * 100000.0 / e.PRSCRPTN_DRUG_TOT_BENES, 1) AS clms_per_100k
    FROM part_d_geo p
    JOIN state_enrollment e ON p.Prscrbr_Geo_Cd = e.BENE_FIPS_CD
    WHERE p.Prscrbr_Geo_Lvl = 'State'
      AND LOWER(p.Gnrc_Name) LIKE '%lenalidomide%'
    GROUP BY p.Prscrbr_Geo_Desc, e.BENE_STATE_ABRVTN, e.PRSCRPTN_DRUG_TOT_BENES
    ORDER BY clms_per_100k DESC
""", conn)

print(f"States in map: {len(df)}")
df.head(10)

States in map: 54


,state,fips,tot_clms,tot_benes,tot_cost,part_d_enrollees,clms_per_100k
0,District of Columbia,11,998,250.0,16436618.0,62769.0,1590.0
1,Virgin Islands,78,156,15.0,2393922.0,11743.0,1328.5
2,Massachusetts,25,11507,2108.0,180933440.0,1148677.0,1001.8
3,Maryland,24,6483,1231.0,103381307.0,734745.0,882.3
4,Georgia,13,12910,2041.0,218351697.0,1473060.0,876.4
5,Connecticut,09,5030,852.0,84449320.0,597393.0,842.0
6,New York,36,25414,4863.0,416899834.0,3119673.0,814.6
7,South Dakota,46,1142,197.0,20217153.0,141206.0,808.7
8,Maine,23,2252,422.0,37257697.0,293697.0,766.8
9,North Carolina,37,12649,2232.0,210623627.0,1714008.0,738.0


## Step 2: Choropleth Map

State-level choropleth showing lenalidomide claims per 100,000 Part D enrollees.
Plotly requires two-digit FIPS codes and `locationmode='USA-states'` does not apply here
since we are using FIPS rather than state abbreviations â€” we use `geojson` mode instead.

In [3]:
fig = px.choropleth(
    df,
    locations='state_abbr',
    locationmode='USA-states',
    color='clms_per_100k',
    scope='usa',
    color_continuous_scale='Blues',
    title='Lenalidomide Claims per 100,000 Medicare Part D Enrollees by State (2023)',
    hover_name='state',
    hover_data={
        'state_abbr': False,
        'tot_clms': True,
        'tot_benes': True,
        'clms_per_100k': True
    },
    labels={'clms_per_100k': 'Claims per 100k'}
)

fig.update_layout(coloraxis_colorbar=dict(title='Claims per 100k'))
fig.show()